In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# read csv
error_data = pd.read_csv("error_data_clean.csv")
filterflow_data = pd.read_csv("filterflow_data.csv")

# switch time type
filterflow_data['timestamp'] = pd.to_datetime(filterflow_data['timestamp'])
error_data['timestamp'] = pd.to_datetime(error_data['timestamp'])

# label "is_error"（±7 hours）
filterflow_data['is_error'] = 0
for idx, row in error_data.iterrows():
    p = row['printer']
    c = row['color']
    t = row['timestamp']
    mask = (
        (filterflow_data['printer'] == p) &
        (filterflow_data['color'] == c) &
        (abs((filterflow_data['timestamp'] - t).dt.total_seconds()) <= 25200)
    )
    filterflow_data.loc[mask, 'is_error'] = 1

# feature engineering
def extract_features(df):
    df = df.copy()
    df['weekday'] = df['timestamp'].dt.weekday
    df['hour'] = df['timestamp'].dt.hour
    df['day_night'] = df['hour'].apply(lambda x: 'day' if 6 <= x <= 18 else 'night')

    stats = df.groupby(['printer', 'color'])['filter_flow'].agg(['mean', 'max', 'min']).reset_index()
    stats.columns = ['printer', 'color', 'avg_flow', 'max_flow', 'min_flow']
    df = df.merge(stats, on=['printer', 'color'], how='left')

    error_stats = df[df['is_error'] == 1].groupby(['printer', 'color'])['filter_flow'].agg(['count', 'mean', 'min', 'max']).reset_index()
    error_stats.columns = ['printer', 'color', 'error_count', 'error_flow_avg', 'error_flow_min', 'error_flow_max']
    df = df.merge(error_stats, on=['printer', 'color'], how='left')

    df[['error_count', 'error_flow_avg', 'error_flow_min', 'error_flow_max']] = \
        df[['error_count', 'error_flow_avg', 'error_flow_min', 'error_flow_max']].fillna(0)

    return df

# build features
filterflow_data = extract_features(filterflow_data)
filterflow_data = pd.get_dummies(filterflow_data, columns=['day_night'], drop_first=True)

# train model
features = ['filter_flow', 'weekday', 'avg_flow', 'max_flow', 'min_flow',
            'error_count', 'error_flow_avg', 'error_flow_min', 'error_flow_max', 'day_night_night']

df_model = filterflow_data.dropna(subset=features + ['is_error'])
X = df_model[features]
y = df_model['is_error']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# evaluate model
y_pred = model.predict(X_test)
if len(model.classes_) == 2:
    y_prob = model.predict_proba(X_test)[:, 1]
    print(classification_report(y_test, y_pred))
    print("ROC AUC Score:", roc_auc_score(y_test, y_prob))
else:
    print("Can't give probability")

# predict probability
if len(model.classes_) == 2:
    df_model['error_probability'] = model.predict_proba(X)[:, 1]
else:
    df_model['error_probability'] = 0.0  # fallback if only one class

# show 
print(df_model[['timestamp', 'printer', 'color', 'filter_flow', 'error_probability']].head())

# sort by probaility des
df_sorted = df_model.sort_values(by='error_probability', ascending=False)
print(df_sorted[['timestamp', 'printer', 'color', 'filter_flow', 'weekday', 'error_probability']].head(30))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

importances = model.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': features,
    'importance': importances
}).sort_values(by='importance', ascending=False)

sns.barplot(x='importance', y='feature', data=feature_importance_df)
plt.title('Feature Importance')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# read csv
error_data = pd.read_csv("error_data_clean.csv")
filterflow_data = pd.read_csv("filterflow_data.csv")

# switch time type
filterflow_data['timestamp'] = pd.to_datetime(filterflow_data['timestamp'])
error_data['timestamp'] = pd.to_datetime(error_data['timestamp'])

# label "is_error"（±7 hours）
error_data['has_nearby_flow'] = 0
for idx, row in error_data.iterrows():
    p = row['printer']
    c = row['color']
    t = row['timestamp']
    mask = (
        (filterflow_data['printer'] == p) &
        (filterflow_data['color'] == c) &
        (abs((filterflow_data['timestamp'] - t).dt.total_seconds()) <= 25200)
    )
    if mask.any():
        error_data.at[idx, 'has_nearby_flow'] = 1

print(error_data['has_nearby_flow'].sum())  

In [ ]:
# how many error actually be matched
error_data['matched'] = 0
for idx, row in error_data.iterrows():
    mask = (
        (filterflow_data['printer'] == row['printer']) &
        (filterflow_data['color'] == row['color']) &
        (abs((filterflow_data['timestamp'] - row['timestamp']).dt.total_seconds()) <= 25200)
    )
    if mask.any():
        error_data.at[idx, 'matched'] = 1

print("Total error：", len(error_data))
print("Match filterflow：", error_data['matched'].sum())
print("Can't match：", (error_data['matched'] == 0).sum())

1. Determine the optimal range (in seconds) for aligning error records with filter flow data, to ensure meaningful correlation analysis.
2. Use statistical metrics and histogram visualization to identify a reasonable time window (e.g., 30s, 60s, 120s) that captures most of the valid pairings without introducing excessive noise.

In [ ]:
diffs = []

for idx, row in error_data.iterrows():
    sub = filterflow_data[
        (filterflow_data['printer'] == row['printer']) &
        (filterflow_data['color'] == row['color'])
    ]
    if sub.empty:
        continue
    sub = sub.copy()
    sub['time_diff'] = (sub['timestamp'] - row['timestamp']).abs()
    diffs.append(sub['time_diff'].min().total_seconds())
    
import numpy as np

print("avg:", np.mean(diffs))
print("median:", np.median(diffs))
print("max:", max(diffs))

import matplotlib.pyplot as plt
plt.hist(diffs, bins=50)
plt.title("distribution of error time")
plt.xlabel("time(sec)）")
plt.ylabel("error amount")
plt.show()